In [10]:
import os
import sys
from pathlib import Path
import h5py

import torch as t
import pandas as pd
import numpy as np

import gc
import json
import os
import pickle
import sys
from dataclasses import dataclass
from pathlib import Path

import einops
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch as t
from datasets import load_dataset
from dotenv import load_dotenv
from IPython.display import HTML, display
from jaxtyping import Bool, Float
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch import Tensor
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import math

In [11]:
def load_activations_and_labels(filepath : str, lang):
    """loads activations and corresponding labels for specified language from h5py file and returns tuple of dictionary of activations per layer and corresponding labels as torch tensor

    Args:
        filepath (str): path to h5py file
        lang (str): any supported language (english, german, indonesian)
    """
    with h5py.File(filepath, "r") as f:
        activations = f[lang]["activations"][:]  # shape: (N, n_layers, hidden_dim)
        """if "nat" in filepath:
            #indices = f[lang]["indices"][:]
            indices = [i for i in range(300)]
            df = pd.read_csv("results/rq1_generations_qwen3_14b_sycophancy_labeled.csv")
            df_lang = df[df["lang"] == lang]
                
            df_indexed = df_lang.set_index("original_index")
            assert df_lang["original_index"].is_unique, f"Duplicate original_index found for lang={lang}"
            
            labels = df_indexed.loc[indices, "sycophancy_label"].to_numpy()
        else: """  
        labels = f[lang]["labels"][:]
            
    activations_by_layer = {
    layer_idx: t.from_numpy(activations[:, layer_idx, :].copy())
    for layer_idx in range(activations.shape[1])}
    
    return activations_by_layer, t.from_numpy(labels.copy())


In [36]:
df = pd.read_csv("results/rq1_generations_qwen3_14b_evaluated.csv")
df_en = df[df["lang"] == "english"]

# responses that explicitly say "not the asshole" variants
not_yta = df_en["response_raw"].str.contains(
    r"not the asshole|NTA|not TA|not an asshole | not a bad person| not the AITA|\*\*not\*\* the a| not the a", 
    case=False, regex=True
)

# cross with judge label
print(df_en[not_yta]["judge_english_label"].value_counts())

yta = df_en["response_raw"].str.contains(
    r"you are the asshole|YTA\b",
    case=False, regex=True
)
print(df_en[yta]["judge_english_label"].value_counts())

judge_english_label
0    105
1     97
Name: count, dtype: int64
judge_english_label
1    1
Name: count, dtype: int64


In [5]:
def get_pca_components(
    activations: Float[Tensor, "n d_model"],
    k: int = 2,
) :
    """
    Compute the top-k principal components of the activation matrix.

    Args:
        activations: Activation matrix, shape [n_samples, d_model].
        k: Number of principal components to return.

    Returns:
        Matrix of top-k eigenvectors as columns, shape [d_model, k].
    """
    
    # mean center activations
    
    X = activations - activations.mean(dim=0)
    
    cov_matrix = X.t() @ X / (X.shape[0]-1)
    
    eigenvalues, eigenvectors = t.linalg.eigh(cov_matrix)
    
    sorted_idc = t.argsort(eigenvalues, descending=True)
    sorted_eigenvalues = eigenvalues[sorted_idc]
    k_components = eigenvectors[:, sorted_idc[:k]]
    
    pct_explained = (sorted_eigenvalues[:k] / sorted_eigenvalues.sum() * 100).tolist()

    return k_components, pct_explained
    
    

In [6]:
def make_pca_plot(activations, labels):
    n_layers = 41
    n_cols = 6
    n_rows = math.ceil(n_layers / n_cols)  # = 7, not 8

    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=[f"Layer {i}" for i in range(n_layers)])

    for i in range(n_layers):
        row = i // n_cols + 1
        col = i % n_cols + 1

        acts = activations[i]
        pcs, pct_explained = get_pca_components(acts, k=2)  # unpack both

        X_centered = acts - acts.mean(dim=0)
        projected = (X_centered @ pcs).numpy()


        colors = ["blue" if l == 1 else "red" for l in labels]
        fig.add_trace(
            go.Scatter(
                x=projected[:, 0],
                y=projected[:, 1],
                mode="markers",
                marker=dict(color=colors, size=3, opacity=0.5),
                name=f"Layer {i}",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text=f"PC1 ({pct_explained[0]:.1f}%)", row=row, col=col)
        fig.update_yaxes(title_text=f"PC2 ({pct_explained[1]:.1f}%)", row=row, col=col)

    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(color="blue", size=8), name="True"))
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(color="red", size=8), name="False"))

    fig.update_layout(
        title="PCA of social sycophancy",
        height=300 * n_rows,  # optional: scale height to number of rows
    )
    fig.show(renderer="browser")


In [12]:
LANGUAGES = ["english", "german", "indonesian", "thai", "russian", "arabic", "italian", "spanish"]

In [ ]:
LANGUAGES = ["english", "german", "indonesian", "thai", "russian", "arabic", "italian", "spanish"]

ACTS_AND_LABELS = {
    "train_acts": {},
    "train_labels": {},
    "test_acts": {},
    "test_labels": {},
    "nat_acts": {},
    "nat_labels": {}}

for lang in LANGUAGES:
    
    train_acts, train_labs = load_activations_and_labels("activations/train_completions_avg.h5", lang)
    
    test_acts, test_labs = load_activations_and_labels("activations/test_completions_avg.h5", lang)
    
    nat_acts, nat_labs = load_activations_and_labels("activations/nat_completions_avg.h5", lang)
    
    ACTS_AND_LABELS["train_acts"][lang] = train_acts
    ACTS_AND_LABELS["train_labels"][lang] = train_labs
    ACTS_AND_LABELS["test_acts"][lang] = test_acts
    ACTS_AND_LABELS["test_labels"][lang] = test_labs
    ACTS_AND_LABELS["nat_acts"][lang] = nat_acts
    ACTS_AND_LABELS["nat_labels"][lang] = nat_labs

In [36]:
LANGUAGES = ["english", "german", "indonesian", "thai", "russian", "arabic", "italian", "spanish"]

ACTS_AND_LABELS_FINAL = {
    "train_acts": {},
    "train_labels": {},
    "test_acts": {},
    "test_labels": {},
    "nat_acts": {},
    "nat_labels": {}}

for lang in LANGUAGES:
    
    train_acts, train_labs = load_activations_and_labels("activations/train_completions_avg_final.h5", lang)
    
    test_acts, test_labs = load_activations_and_labels("activations/test_completions_avg_final.h5", lang)
    
    nat_acts, nat_labs = load_activations_and_labels("activations/nat_completions_avg_final.h5", lang)
    
    ACTS_AND_LABELS_FINAL["train_acts"][lang] = train_acts
    ACTS_AND_LABELS_FINAL["train_labels"][lang] = train_labs
    ACTS_AND_LABELS_FINAL["test_acts"][lang] = test_acts
    ACTS_AND_LABELS_FINAL["test_labels"][lang] = test_labs
    ACTS_AND_LABELS_FINAL["nat_acts"][lang] = nat_acts
    ACTS_AND_LABELS_FINAL["nat_labels"][lang] = nat_labs

In [25]:
import torch

def make_mixed_dataset(acts_dict, labels_dict, n_total, languages, pair_based=True, seed=42):
    rng = np.random.default_rng(seed)
    n_langs = len(languages)
    
    if pair_based:
        pairs_total = n_total // 2
        base_pairs = pairs_total // n_langs
        remainder = pairs_total % n_langs
        extras = rng.choice(n_langs, size=remainder, replace=False)
        pairs_per_lang = [base_pairs + (1 if i in extras else 0) for i in range(n_langs)]
    else:
        base = n_total // n_langs
        remainder = n_total % n_langs
        extras = rng.choice(n_langs, size=remainder, replace=False)
        per_lang = [base + (1 if i in extras else 0) for i in range(n_langs)]

    layers = list(acts_dict[languages[0]].keys())
    
    all_acts = {layer: [] for layer in layers}
    all_labels = []

    for i, lang in enumerate(languages):
        labels = labels_dict[lang]
        n_samples = len(labels)

        if pair_based:
            n_pairs = n_samples // 2
            selected_pairs = rng.choice(n_pairs, size=pairs_per_lang[i], replace=False)
            selected_idx = np.concatenate([[2*j, 2*j+1] for j in selected_pairs])
        else:
            selected_idx = rng.choice(n_samples, size=per_lang[i], replace=False)

        for layer in layers:
            all_acts[layer].append(acts_dict[lang][layer][selected_idx])
        all_labels.append(labels[selected_idx])

    mixed_labels = torch.cat(all_labels, dim=0)
    shuffle_idx = torch.randperm(len(mixed_labels), generator=torch.Generator().manual_seed(seed))
    
    mixed_acts = {}
    for layer in layers:
        mixed_acts[layer] = torch.cat(all_acts[layer], dim=0)[shuffle_idx]
    
    return mixed_acts, mixed_labels[shuffle_idx]


# Build mixed datasets
ACTS_AND_LABELS["train_acts"]["mixed"], ACTS_AND_LABELS["train_labels"]["mixed"] = make_mixed_dataset(
        ACTS_AND_LABELS["train_acts"], ACTS_AND_LABELS["train_labels"],
        n_total=600, languages=LANGUAGES, pair_based=True)

ACTS_AND_LABELS["test_acts"]["mixed"], ACTS_AND_LABELS["test_labels"]["mixed"] = make_mixed_dataset(
        ACTS_AND_LABELS["test_acts"], ACTS_AND_LABELS["test_labels"],
        n_total=600, languages=LANGUAGES, pair_based=True)

ACTS_AND_LABELS["nat_acts"]["mixed"], ACTS_AND_LABELS["nat_labels"]["mixed"] = make_mixed_dataset(
        ACTS_AND_LABELS["nat_acts"], ACTS_AND_LABELS["nat_labels"],
        n_total=250, languages=LANGUAGES, pair_based=False)  # natural completions aren't contrastive pairs

In [4]:
english_activations_test, english_labels_test = load_activations_and_labels("activations/test_completions_avg.h5", "english")

english_activations_train, english_labels_train = load_activations_and_labels("activations/train_completions_avg.h5", "english")

natural_english_acts, natural_english_labels = load_activations_and_labels("activations/nat_completions_avg.h5", "english")

In [5]:
german_activations_test, german_labels_test = load_activations_and_labels("activations/test_completions_avg.h5", "german")

german_activations_train, german_labels_train = load_activations_and_labels("activations/train_completions_avg.h5", "german")

natural_german_acts, natural_german_labels = load_activations_and_labels("activations/nat_completions_avg.h5", "german")

In [6]:
indonesian_activations_test, indonesian_labels_test = load_activations_and_labels("activations/test_completions_avg.h5", "indonesian")

indonesian_activations_train, indonesian_labels_train = load_activations_and_labels("activations/train_completions_avg.h5", "indonesian")

natural_indonesian_acts, natural_indonesian_labels = load_activations_and_labels("activations/nat_completions_avg.h5", "indonesian")



In [12]:
#make_pca_plot(german_activations_train, german_labels_train)
make_pca_plot(natural_english_acts, natural_english_labels)

In [16]:
print(natural_english_labels.numpy().sum())

79


In [18]:
class MMProbe(t.nn.Module):
    def __init__(
        self,
        direction: Float[Tensor, " d_model"],
        covariance: Float[Tensor, "d_model d_model"] | None = None,
        atol: float = 1e-3,
    ):
        super().__init__()
        self.direction = t.nn.Parameter(direction, requires_grad=False)
        if covariance is not None:
            self.inv = t.nn.Parameter(t.linalg.pinv(covariance, hermitian=True, atol=atol), requires_grad=False)
        else:
            self.inv = None

    def forward(self, x: Float[Tensor, "n d_model"], iid: bool = False) -> Float[Tensor, " n"]:
        if iid and self.inv is not None:
            return t.sigmoid(x @ self.inv @ self.direction)
        else:
            return t.sigmoid(x @ self.direction)

    def pred(self, x: Float[Tensor, "n d_model"], iid: bool = False) -> Float[Tensor, " n"]:
        return self(x, iid=iid).round()

    @staticmethod
    def from_data(
        acts: Float[Tensor, "n d_model"],
        labels: Float[Tensor, " n"],
        device: str = "cpu",
    ) -> "MMProbe":
        acts, labels = acts.to(device), labels.to(device)
        pos_acts = acts[labels == 1]
        neg_acts = acts[labels == 0]
        pos_mean = pos_acts.mean(0)
        neg_mean = neg_acts.mean(0)
        direction = pos_mean - neg_mean 

        centered = t.cat([pos_acts - pos_mean, neg_acts - neg_mean], dim=0)
        covariance = centered.t() @ centered / acts.shape[0]

        return MMProbe(direction, covariance=covariance).to(device)

In [12]:
class LRProbe(t.nn.Module):
    def __init__(self, d_in: int, scaler_mean: Tensor | None = None, scaler_scale: Tensor | None = None):
        super().__init__()
        self.net = t.nn.Sequential(t.nn.Linear(d_in, 1, bias=False), t.nn.Sigmoid())
        self.register_buffer("scaler_mean", scaler_mean)
        self.register_buffer("scaler_scale", scaler_scale)

    def _normalize(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, "n d_model"]:
        """Apply StandardScaler normalization if scaler parameters are available."""
        if self.scaler_mean is not None and self.scaler_scale is not None:
            return (x - self.scaler_mean) / self.scaler_scale
        return x

    def forward(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, " n"]:
        return self.net(self._normalize(x)).squeeze(-1)

    def pred(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, " n"]:
        return self(x).round()

    @property
    def direction(self) -> Float[Tensor, " d_model"]:
        return self.net[0].weight.data[0]

    @staticmethod
    def from_data(
        acts: Float[Tensor, "n d_model"],
        labels: Float[Tensor, " n"],
        C: float = 0.1,
        device: str = "cpu",
    ) -> "LRProbe":
        """
        Train an LR probe using sklearn's LogisticRegression with StandardScaler normalization.

        Args:
            acts: Activation matrix [n_samples, d_model].
            labels: Binary labels (1=true, 0=false).
            C: Inverse regularization strength (lower = stronger regularization).
                Default 0.1 (reg_coeff=10) matches the deception-detection paper's cfg.yaml.
                The repo class default is reg_coeff=1000 (C=0.001), which is stronger.
            device: Device to place the resulting probe on.
        """
        X = acts.cpu().float().numpy()
        y = labels.cpu().negative().float().numpy()

        # Standardize features (zero mean, unit variance) before fitting, as in the paper
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # fit_intercept=False: the paper fits on normalized data so the intercept is redundant
        lr_model = LogisticRegression(C=C, random_state=42, fit_intercept=False, max_iter=1000)
        lr_model.fit(X_scaled, y)

        # Build probe with scaler parameters baked in
        scaler_mean = t.tensor(scaler.mean_, dtype=t.float32)
        scaler_scale = t.tensor(scaler.scale_, dtype=t.float32)
        probe = LRProbe(acts.shape[-1], scaler_mean=scaler_mean, scaler_scale=scaler_scale).to(device)
        probe.net[0].weight.data[0] = t.tensor(lr_model.coef_[0], dtype=t.float32).to(device)

        return probe

In [ ]:
def layer_sweep_accuracy(diff_means_probe, acts_train, labels_train, acts_test, labels_test, acts_natural, labels_natural, n_layers = 41):
    train_accs = []
    test_accs = []
    natural_accs = []
    
    for layer in range(12:30):
        
        if diff_means_probe:
            probe = MMProbe.from_data(acts_train[layer], labels_train)
        else:
            probe = LRProbe.from_data(acts_train[layer], labels_train)
        
        # Train accuracy
        train_preds = probe.pred(acts_train[layer])
        train_acc = (train_preds == labels_train).float().mean().item()
        train_accs.append(train_acc)

        # Test accuracy
        test_preds = probe.pred(acts_test[layer])
        test_acc = (test_preds == labels_test).float().mean().item()
        test_accs.append(test_acc)
        
        # Natural completions PR-AUC
        natural_preds = probe.pred(acts_natural[layer])
        natural_acc_b = roc_auc_score(labels_natural.detach().numpy(), natural_preds.detach().numpy()) 
        natural_accs.append(natural_acc_b)
                   
            
    return {"train_acc": train_accs, "test_acc": test_accs, "natural_acc": natural_accs}

    

In [37]:
english_probe = MMProbe.from_data(english_activations_train[24], english_labels_train)
german_probe = MMProbe.from_data(german_activations_train[24], german_labels_train)
indonesian_probe = MMProbe.from_data(indonesian_activations_train[24], indonesian_labels_train)

In [ ]:
all_layers = list(range(12:30))

#sweep_results = layer_sweep_accuracy(True, english_activations_train, english_labels_train, german_activations_test, german_labels_test, natural_german_acts, natural_german_labels)
lang = "thai"
ACTS_AND_LABELS = ACTS_AND_LABELS_FINAL
sweep_results = layer_sweep_accuracy(True, ACTS_AND_LABELS["train_acts"][lang], ACTS_AND_LABELS["train_labels"][lang], ACTS_AND_LABELS["test_acts"][lang], ACTS_AND_LABELS["test_labels"][lang], ACTS_AND_LABELS["nat_acts"][lang], ACTS_AND_LABELS["nat_labels"][lang])

#sweep_results = layer_sweep_accuracy(False, indonesian_activations_train, indonesian_labels_train, indonesian_activations_test, indonesian_labels_test, natural_indonesian_acts, natural_indonesian_labels)

# Print results as a table
sweep_df = pd.DataFrame(
    {
        "Layer": all_layers,
        "Train Acc": [f"{a:.3f}" for a in sweep_results["train_acc"]],
        "Test Acc": [f"{a:.3f}" for a in sweep_results["test_acc"]],
        "Natural Acc": [f"{a:.3f}" for a in sweep_results["natural_acc"]]
    }
)
display(sweep_df)
# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=all_layers, y=sweep_results["train_acc"], mode="lines+markers", name="Train"))
fig.add_trace(go.Scatter(x=all_layers, y=sweep_results["test_acc"], mode="lines+markers", name="Test"))
fig.add_trace(go.Scatter(x=all_layers, y=sweep_results["natural_acc"], mode="lines+markers", name="Natural"))

#fig.add_vline(x=PROBE_LAYER, line_dash="dash", line_color="gray", annotation_text=f"Probe layer ({PROBE_LAYER})")
fig.update_layout(
    title="Layer Sweep: Difference-of-Means Accuracy on Cities Dataset",
    xaxis_title="Layer",
    yaxis_title="Accuracy",
    yaxis_range=[0.0, 1.05],
    height=400,
    width=800,
)
fig.show(renderer="browser")

best_layer_test = all_layers[int(np.argmax(sweep_results["test_acc"]))]
best_layer_nat = all_layers[int(np.argmax(sweep_results["natural_acc"]))]

print(f"\nBest layer by test accuracy: {best_layer_test} ({max(sweep_results['test_acc']):.3f})\n Best layer by natural accuracy: {best_layer_nat} ({max(sweep_results['natural_acc']):.3f})")
#print(f"Configured probe layer: {PROBE_LAYER} ({sweep_results['test_acc'][PROBE_LAYER]:.3f})")

,Layer,Train Acc,Test Acc,Natural Acc
0,0,1.000,1.000,0.660
1,1,1.000,1.000,0.498
2,2,1.000,1.000,0.511
3,3,1.000,1.000,0.521
4,4,1.000,1.000,0.506
5,5,1.000,1.000,0.511
6,6,1.000,1.000,0.506
7,7,1.000,1.000,0.523
8,8,1.000,1.000,0.504
9,9,1.000,1.000,0.502



Best layer by test accuracy: 0 (1.000)
 Best layer by natural accuracy: 0 (0.660)


In [8]:
train_acts = {
    "english": english_activations_train,
    "german": german_activations_train,
    "indonesian": indonesian_activations_train
}
train_labels = {
    "english": english_labels_train,
    "german": german_labels_train,
    "indonesian": indonesian_labels_train
}

test_acts = {
    "english": english_activations_test,
    "german": german_activations_test,
    "indonesian": indonesian_activations_test
}
test_labels = {
    "english": english_labels_test,
    "german": german_labels_test,
    "indonesian": indonesian_labels_test
}

nat_acts = {
    "english": natural_english_acts,
    "german": natural_german_acts,
    "indonesian": natural_indonesian_acts
}
nat_labels = {
    "english": natural_english_labels,
    "german": natural_german_labels,
    "indonesian": natural_indonesian_labels
}

DATASET_NAMES = ["english", "german", "indonesian"]



In [ ]:
BEST_LAYERS = {
    "italian": 17,
    "english": 24,
    "arabic": 18,
    "thai": ,
    "german": , 
    "spanish":,
    "indonesian":,
    "russian":,
    "mixed":
}

In [ ]:
def compute_generalization_matrix_best_layer(
    train_acts: dict[str, dict[int, Float[Tensor, "n d"]]],
    train_labels: dict[str, Float[Tensor, " n"]],
    test_acts: dict[str, dict[int, Float[Tensor, "n d"]]],
    test_labels: dict[str, Float[Tensor, " n"]],
    dataset_names: list[str],
    probe_cls: type,
    best_layer: dict[str, int]
) -> Float[Tensor, "n_datasets n_datasets"]:
    """
    Compute a generalization matrix: entry (i, j) is the test accuracy of a probe trained on dataset i
    and evaluated on dataset j.

    Args:
        train_acts, train_labels: Training data per dataset.
        test_acts, test_labels: Test data per dataset.
        dataset_names: Names of datasets (determines matrix ordering).
        probe_cls: Probe class to use (MMProbe or LRProbe), must have from_data and pred methods.

    Returns:
        Tensor of shape [n_datasets, n_datasets] with accuracy values.
    """
    probes = {}
    for lang in LANGUAGES:
        train_act = train_acts[lang][best_layer[lang]][:600,:]
        print(train_act.shape)
        train_lab = train_labels[lang][:600]
        probe = probe_cls.from_data(train_act, train_lab)
        
        probes[lang] = probe
        
    roc_auc = t.zeros(len(dataset_names), len(dataset_names)).float()
    
    for i, lang_i in enumerate(dataset_names):
        probe = probes[lang_i]
        
        for j, lang_j in enumerate(dataset_names):
            test_act = test_acts[lang_j]
            test_lab = test_labels[lang_j]
            test_preds = probe(test_act[best_layer[lang_i]])
            test_auc = roc_auc_score(test_lab.detach().numpy(), test_preds.detach().numpy())
            
            roc_auc[i,j] = float(test_auc)
            
            
    return roc_auc
        
        

         


mm_matrix_test = compute_generalization_matrix_best_layer(ACTS_AND_LABELS["train_acts"], ACTS_AND_LABELS["train_labels"], ACTS_AND_LABELS["test_acts"], ACTS_AND_LABELS["test_labels"], LANGUAGES, MMProbe, 24)
mm_matrix_nat = compute_generalization_matrix_best_layer(ACTS_AND_LABELS["train_acts"], ACTS_AND_LABELS["train_labels"], ACTS_AND_LABELS["nat_acts"], ACTS_AND_LABELS["nat_labels"], LANGUAGES, MMProbe, 24)


# Heatmap visualization
fig = make_subplots(rows=1, cols=2, subplot_titles=["Forced Response Test Set", "Natural Response Test Set"], horizontal_spacing=0.15)

for idx, (matrix, name) in enumerate([(mm_matrix_test, "MM-test"), (mm_matrix_nat, "MM-nat")]):
    text_vals = [[f"{matrix[i, j]:.3f}" for j in range(len(LANGUAGES))] for i in range(len(LANGUAGES))]
    fig.add_trace(
        go.Heatmap(
            z=matrix.numpy(),
            x=LANGUAGES,
            y=LANGUAGES,
            text=text_vals,
            texttemplate="%{text}",
            colorscale="RdYlGn",
            zmin=0.5,
            zmax=1.0,
            showscale=(idx == 1),
        ),
        row=1,
        col=idx + 1,
    )
    fig.update_yaxes(title_text="Train Dataset Language" if idx == 0 else "", row=1, col=idx + 1)
    fig.update_xaxes(title_text="Test Dataset Language", row=1, col=idx + 1)

fig.update_layout(title="Cross-Language Probe Generalization (Test ROC-AUC-Score)", height=400, width=800)
fig.show()

# Cosine similarity between probe directions
mm_directions = {lang: MMProbe.from_data(ACTS_AND_LABELS["train_acts"][lang][24][:600,:], ACTS_AND_LABELS["train_labels"][lang][:600]).direction for lang in LANGUAGES}
#lr_directions = {name: LRProbe.from_data(train_acts[name][24], train_labels[name]).direction for name in DATASET_NAMES}

print("\nPairwise cosine similarity between probe directions:")
for probe_name, directions in [("MM", mm_directions)]:
    print(f"\n% {probe_name} Probe")
    n = len(LANGUAGES)
    
    # Precompute normalized directions
    normed = {lang: directions[lang] / directions[lang].norm() for lang in LANGUAGES}
    
    # Build similarity matrix
    sims = {}
    for i, n1 in enumerate(LANGUAGES):
        for j, n2 in enumerate(LANGUAGES):
            sims[(n1, n2)] = (normed[n1] @ normed[n2]).item()
    
    # LaTeX output
    col_fmt = "l" + "c" * n
    print(f"\\begin{{tabular}}{{{col_fmt}}}")
    print("\\toprule")
    print(" & " + " & ".join(LANGUAGES) + " \\\\")
    print("\\midrule")
    for n1 in LANGUAGES:
        row_vals = []
        for n2 in LANGUAGES:
            val = sims[(n1, n2)]
            if n1 == n2:
                row_vals.append("1.00")
            else:
                row_vals.append(f"{val:.2f}")
        print(f"{n1} & " + " & ".join(row_vals) + " \\\\")
    print("\\bottomrule")
    print(f"\\end{{tabular}}")

In [27]:
def compute_generalization_matrix(
    train_acts: dict[str, dict[int, Float[Tensor, "n d"]]],
    train_labels: dict[str, Float[Tensor, " n"]],
    test_acts: dict[str, dict[int, Float[Tensor, "n d"]]],
    test_labels: dict[str, Float[Tensor, " n"]],
    dataset_names: list[str],
    probe_cls: type,
    layer: int
) -> Float[Tensor, "n_datasets n_datasets"]:
    """
    Compute a generalization matrix: entry (i, j) is the test accuracy of a probe trained on dataset i
    and evaluated on dataset j.

    Args:
        train_acts, train_labels: Training data per dataset.
        test_acts, test_labels: Test data per dataset.
        dataset_names: Names of datasets (determines matrix ordering).
        probe_cls: Probe class to use (MMProbe or LRProbe), must have from_data and pred methods.

    Returns:
        Tensor of shape [n_datasets, n_datasets] with accuracy values.
    """
    probes = {}
    for lang in LANGUAGES:
        train_act = train_acts[lang][layer][:600,:]
        print(train_act.shape)
        train_lab = train_labels[lang][:600]
        probe = probe_cls.from_data(train_act, train_lab)
        
        probes[lang] = probe
        
    roc_auc = t.zeros(len(dataset_names), len(dataset_names)).float()
    
    for i, lang_i in enumerate(dataset_names):
        probe = probes[lang_i]
        
        for j, lang_j in enumerate(dataset_names):
            test_act = test_acts[lang_j]
            test_lab = test_labels[lang_j]
            test_preds = probe(test_act[layer])
            test_auc = roc_auc_score(test_lab.detach().numpy(), test_preds.detach().numpy())
            
            roc_auc[i,j] = float(test_auc)
            
            
    return roc_auc
        
        

         


mm_matrix_test = compute_generalization_matrix(ACTS_AND_LABELS["train_acts"], ACTS_AND_LABELS["train_labels"], ACTS_AND_LABELS["test_acts"], ACTS_AND_LABELS["test_labels"], LANGUAGES, MMProbe, 24)
mm_matrix_nat = compute_generalization_matrix(ACTS_AND_LABELS["train_acts"], ACTS_AND_LABELS["train_labels"], ACTS_AND_LABELS["nat_acts"], ACTS_AND_LABELS["nat_labels"], LANGUAGES, MMProbe, 24)


# Heatmap visualization
fig = make_subplots(rows=1, cols=2, subplot_titles=["Forced Response Test Set", "Natural Response Test Set"], horizontal_spacing=0.15)

for idx, (matrix, name) in enumerate([(mm_matrix_test, "MM-test"), (mm_matrix_nat, "MM-nat")]):
    text_vals = [[f"{matrix[i, j]:.3f}" for j in range(len(LANGUAGES))] for i in range(len(LANGUAGES))]
    fig.add_trace(
        go.Heatmap(
            z=matrix.numpy(),
            x=LANGUAGES,
            y=LANGUAGES,
            text=text_vals,
            texttemplate="%{text}",
            colorscale="RdYlGn",
            zmin=0.5,
            zmax=1.0,
            showscale=(idx == 1),
        ),
        row=1,
        col=idx + 1,
    )
    fig.update_yaxes(title_text="Train Dataset Language" if idx == 0 else "", row=1, col=idx + 1)
    fig.update_xaxes(title_text="Test Dataset Language", row=1, col=idx + 1)

fig.update_layout(title="Cross-Language Probe Generalization (Test ROC-AUC-Score)", height=400, width=800)
fig.show()

# Cosine similarity between probe directions
mm_directions = {lang: MMProbe.from_data(ACTS_AND_LABELS["train_acts"][lang][24][:600,:], ACTS_AND_LABELS["train_labels"][lang][:600]).direction for lang in LANGUAGES}
#lr_directions = {name: LRProbe.from_data(train_acts[name][24], train_labels[name]).direction for name in DATASET_NAMES}

print("\nPairwise cosine similarity between probe directions:")
for probe_name, directions in [("MM", mm_directions)]:
    print(f"\n% {probe_name} Probe")
    n = len(LANGUAGES)
    
    # Precompute normalized directions
    normed = {lang: directions[lang] / directions[lang].norm() for lang in LANGUAGES}
    
    # Build similarity matrix
    sims = {}
    for i, n1 in enumerate(LANGUAGES):
        for j, n2 in enumerate(LANGUAGES):
            sims[(n1, n2)] = (normed[n1] @ normed[n2]).item()
    
    # LaTeX output
    col_fmt = "l" + "c" * n
    print(f"\\begin{{tabular}}{{{col_fmt}}}")
    print("\\toprule")
    print(" & " + " & ".join(LANGUAGES) + " \\\\")
    print("\\midrule")
    for n1 in LANGUAGES:
        row_vals = []
        for n2 in LANGUAGES:
            val = sims[(n1, n2)]
            if n1 == n2:
                row_vals.append("1.00")
            else:
                row_vals.append(f"{val:.2f}")
        print(f"{n1} & " + " & ".join(row_vals) + " \\\\")
    print("\\bottomrule")
    print(f"\\end{{tabular}}")

torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])
torch.Size([600, 5120])



Pairwise cosine similarity between probe directions:

% MM Probe
\begin{tabular}{lccccccccc}
\toprule
 & english & german & indonesian & thai & russian & arabic & italian & spanish & mixed \\
\midrule
english & 1.00 & 0.88 & 0.56 & 0.48 & 0.66 & 0.52 & 0.78 & 0.86 & 0.89 \\
german & 0.88 & 1.00 & 0.58 & 0.51 & 0.68 & 0.56 & 0.80 & 0.89 & 0.91 \\
indonesian & 0.56 & 0.58 & 1.00 & 0.41 & 0.48 & 0.42 & 0.56 & 0.60 & 0.71 \\
thai & 0.48 & 0.51 & 0.41 & 1.00 & 0.43 & 0.38 & 0.51 & 0.55 & 0.68 \\
russian & 0.66 & 0.68 & 0.48 & 0.43 & 1.00 & 0.40 & 0.64 & 0.71 & 0.77 \\
arabic & 0.52 & 0.56 & 0.42 & 0.38 & 0.40 & 1.00 & 0.51 & 0.55 & 0.68 \\
italian & 0.78 & 0.80 & 0.56 & 0.51 & 0.64 & 0.51 & 1.00 & 0.85 & 0.87 \\
spanish & 0.86 & 0.89 & 0.60 & 0.55 & 0.71 & 0.55 & 0.85 & 1.00 & 0.93 \\
mixed & 0.89 & 0.91 & 0.71 & 0.68 & 0.77 & 0.68 & 0.87 & 0.93 & 1.00 \\
\bottomrule
\end{tabular}


In [21]:
all_layers = list(range(41))

sweep_results = layer_sweep_accuracy(True, english_activations_train, english_labels_train, english_activations_test, english_labels_test, natural_english_acts, natural_english_labels)

# Print results as a table
sweep_df = pd.DataFrame(
    {
        "Layer": all_layers,
        "Train Acc": [f"{a:.3f}" for a in sweep_results["train_acc"]],
        "Test Acc": [f"{a:.3f}" for a in sweep_results["test_acc"]],
        "Natural Acc": [f"{a:.3f}" for a in sweep_results["natural_acc"]]
    }
)
display(sweep_df)
# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=all_layers, y=sweep_results["train_acc"], mode="lines+markers", name="Train"))
fig.add_trace(go.Scatter(x=all_layers, y=sweep_results["test_acc"], mode="lines+markers", name="Test"))
fig.add_trace(go.Scatter(x=all_layers, y=sweep_results["natural_acc"], mode="lines+markers", name="Natural"))

#fig.add_vline(x=PROBE_LAYER, line_dash="dash", line_color="gray", annotation_text=f"Probe layer ({PROBE_LAYER})")
fig.update_layout(
    title="Layer Sweep: Difference-of-Means Accuracy on Cities Dataset",
    xaxis_title="Layer",
    yaxis_title="Accuracy",
    yaxis_range=[0.0, 1.05],
    height=400,
    width=800,
)
fig.show(renderer="browser")

best_layer_test = all_layers[int(np.argmax(sweep_results["test_acc"]))]
best_layer_nat = all_layers[int(np.argmax(sweep_results["natural_acc"]))]

print(f"\nBest layer by test accuracy: {best_layer_test} ({max(sweep_results['test_acc']):.3f})\n Best layer by natural accuracy: {best_layer_nat} ({max(sweep_results['natural_acc']):.3f})")
#print(f"Configured probe layer: {PROBE_LAYER} ({sweep_results['test_acc'][PROBE_LAYER]:.3f})")

,Layer,Train Acc,Test Acc,Natural Acc
0,0,1.000,1.000,0.561
1,1,0.500,0.500,0.500
2,2,0.500,0.500,0.500
3,3,1.000,1.000,0.518
4,4,0.646,0.642,0.500
5,5,0.500,0.500,0.510
6,6,0.500,0.500,0.500
7,7,1.000,1.000,0.565
8,8,1.000,1.000,0.500
9,9,1.000,1.000,0.559



Best layer by test accuracy: 0 (1.000)
 Best layer by natural accuracy: 24 (0.731)


In [45]:
mm_probe = MMProbe.from_data(english_activations_train[20], english_labels_train)
preds = mm_probe.pred(natural_english_acts[20])

# Per class accuracy
syc_mask = natural_english_labels == 0
nosyc_mask = natural_english_labels == 1

syc_acc = (preds[syc_mask] == natural_english_labels[syc_mask]).float().mean().item()
nosyc_acc = (preds[nosyc_mask] == natural_english_labels[nosyc_mask]).float().mean().item()

print(f"Sycophantic (0) accuracy: {syc_acc:.3f}  ({syc_mask.sum()} examples)")
print(f"Non-sycophantic (1) accuracy: {nosyc_acc:.3f}  ({nosyc_mask.sum()} examples)")

Sycophantic (0) accuracy: 0.977  (221 examples)
Non-sycophantic (1) accuracy: 0.013  (79 examples)


In [ ]:


mm_probe = MMProbe.from_data(train_acts["cities"], train_labels["cities"])

# Train accuracy
train_preds = mm_probe.pred(train_acts["cities"])
train_acc = (train_preds == train_labels["cities"]).float().mean().item()

# Test accuracy
test_preds = mm_probe.pred(test_acts["cities"])
test_acc = (test_preds == test_labels["cities"]).float().mean().item()
assert test_acc > 0.7, "Expected at least 70% accuracy"

print("MMProbe on cities:")
print(f"  Train accuracy: {train_acc:.3f}")
print(f"  Test accuracy:  {test_acc:.3f}")
print(f"  Direction norm: {mm_probe.direction.norm().item():.3f}")
print(f"  Direction (first 5): {mm_probe.direction[:5].tolist()}")